# Multi-Signal Movie Recommender 

## 1. Install Dependencies

In [1]:
!pip install -q faiss-cpu kaggle
# Note: lightfm builds a small C extension on install. If it fails on your platform,
# set USE_LIGHTFM = False below and the notebook will fall back to a content-only path.

# !apt-get update -qq && apt-get install -y -qq build-essential python3-dev && pip install -q "setuptools<82" "Cython<3" "lightfm==1.17" --no-build-isolation

!pip install -q lightfm-next

# scikit-surprise is intentionally NOT installed here. v2 trained a Surprise SVD whose
# output (item_latent_factors) was never used by any downstream scoring or evaluation
# function. Removed rather than left as dead code.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.7 MB/s eta 0:00:00


## 2. Download Data from Kaggle

In [2]:
# from google.colab import files
# files.upload()  # upload kaggle.json

# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d sheikhmuneebahmed115/advanced-imdb
!unzip -q advanced-imdb.zip -d data
!ls data/

Dataset URL: https://www.kaggle.com/datasets/sheikhmuneebahmed115/advanced-imdb
License(s): CC0-1.0
100% 73.1M/73.1M [00:00<00:00, 83.4MB/s]

advanced-imdb.csv  advanced-imdb-train.csv  ratings.csv


## 3. Imports

In [3]:
import numpy as np
import pandas as pd
import faiss
import random
import warnings
import ast
from difflib import get_close_matches
from collections import defaultdict, Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix, hstack as sp_hstack, vstack as sp_vstack

# Bridges the 8K-rated / 200K-cold-start gap via shared content features
USE_LIGHTFM = True
try:
    from lightfm import LightFM
    from lightfm.evaluation import auc_score
except ImportError:
    print('lightfm not installed - set USE_LIGHTFM = False, hybrid rerank will skip the LightFM term.')
    USE_LIGHTFM = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 120)
print('Imports ready')

Imports ready


## 4. Load Raw Data

In [4]:
movies_8k   = pd.read_csv('data/advanced-imdb-train.csv')
movies_200k = pd.read_csv('data/advanced-imdb.csv')
ratings     = pd.read_csv('data/ratings.csv')

# Align the genre column name (200k uses 'genres', 8k uses 'genres_y')
movies_200k = movies_200k.rename(columns={'genres': 'genres_y'})

print(f'movies_8k  : {movies_8k.shape}')
print(f'movies_200k: {movies_200k.shape}')
print(f'ratings    : {ratings.shape}')
print(movies_8k[['title', 'vote_average', 'vote_count']].head(3))

movies_8k  : (8248, 33)
movies_200k: (235715, 32)
ratings    : (100836, 4)
                          title  vote_average  vote_count
0       The Great Train Robbery         7.000         547
1         The Birth of a Nation         6.034         482
2  20,000 Leagues Under the Sea         6.300          48


## 5. IMDB Weighted Rating 

In [5]:
def add_imdb_weighted_rating(df, m, C):
    v = df['vote_count'].fillna(0)
    R = df['vote_average'].fillna(0)
    df = df.copy()
    df['weighted_rating'] = (v / (v + m)) * R + (m / (v + m)) * C
    return df

# m and C are computed on the UNION of both catalogs so the threshold reflects
# the whole 282K-movie universe, not just the 8K subset.
combined_votes = pd.concat([movies_8k['vote_count'], movies_200k['vote_count']], ignore_index=True)
combined_avg   = pd.concat([movies_8k['vote_average'], movies_200k['vote_average']], ignore_index=True)

M_THRESHOLD = combined_votes.quantile(0.80)   # top-20%-by-votes reliability bar
C_GLOBAL_MEAN = combined_avg.mean()

movies_8k   = add_imdb_weighted_rating(movies_8k, M_THRESHOLD, C_GLOBAL_MEAN)
movies_200k = add_imdb_weighted_rating(movies_200k, M_THRESHOLD, C_GLOBAL_MEAN)

print(f'IMDB WR params -> m={M_THRESHOLD:.1f} votes, C={C_GLOBAL_MEAN:.2f}')
print(movies_8k[['title', 'vote_average', 'vote_count', 'weighted_rating']].sort_values('weighted_rating', ascending=False).head(5))

IMDB WR params -> m=12.0 votes, C=3.62
                         title  vote_average  vote_count  weighted_rating
916              The Godfather         8.707       18677         8.703732
2961  The Shawshank Redemption         8.702       24649         8.699526
982      The Godfather Part II         8.591       11293         8.585720
2751          Schindler's List         8.573       14594         8.568928
485               12 Angry Men         8.540        7658         8.532297


## 6. Real per-movie rating aggregates from `ratings.csv` (8K only)


In [6]:
movie_rating_features = ratings.groupby('movieId').agg(
    avg_rating   = ('rating', 'mean'),
    rating_count = ('rating', 'count'),
    rating_std   = ('rating', 'std'),
).reset_index()

m_bayes = movie_rating_features['rating_count'].mean()
C_bayes = movie_rating_features['avg_rating'].mean()

movie_rating_features['bayesian_rating'] = (
    (movie_rating_features['rating_count'] * movie_rating_features['avg_rating'] + m_bayes * C_bayes)
    / (movie_rating_features['rating_count'] + m_bayes)
)

movies_8k = movies_8k.merge(movie_rating_features, on='movieId', how='left')
movies_8k['avg_rating']      = movies_8k['avg_rating'].fillna(movies_8k['avg_rating'].mean())
movies_8k['bayesian_rating'] = movies_8k['bayesian_rating'].fillna(movies_8k['bayesian_rating'].mean())
movies_8k['rating_std']      = movies_8k['rating_std'].fillna(0)
movies_8k['rating_count']    = movies_8k['rating_count'].fillna(0)

print(f'Rating aggregates merged (diagnostic only, not used in scoring). movies_8k shape: {movies_8k.shape}')

Rating aggregates merged (diagnostic only, not used in scoring). movies_8k shape: (8248, 38)


## 7. Extract Release Year + Decade (both catalogs)

In [7]:
def extract_year(df):
    df = df.copy()
    df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year.astype(float)
    df['release_year'] = df['release_year'].fillna(df['release_year'].median())
    df['decade'] = (df['release_year'] // 10).astype(float)
    return df

movies_8k   = extract_year(movies_8k)
movies_200k = extract_year(movies_200k)
print('Year/decade extracted for both catalogs')

Year/decade extracted for both catalogs


## 8. Text Feature Builders (unchanged logic, applied to both catalogs)

In [8]:
def build_genre_text(row):
    return ' '.join([str(row.get('genres_y', '')), str(row.get('keywords', ''))])

def build_people_text(row):
    return ' '.join([str(row.get('cast', ''))[:300], str(row.get('directors', '')), str(row.get('writers', ''))[:100]])

def build_overview_text(row):
    return str(row.get('overview', ''))[:500]

def build_title_text(row):
    return str(row.get('title', ''))

def build_language_text(row):
    lang = str(row.get('original_language', 'unknown')).strip().lower()
    if lang in ('', 'nan', 'none', 'unknown'):
        lang = 'xx'
    return ' '.join([lang] * 5)

for df in (movies_8k, movies_200k):
    df['genre_text']    = df.apply(build_genre_text, axis=1)
    df['people_text']   = df.apply(build_people_text, axis=1)
    df['overview_text'] = df.apply(build_overview_text, axis=1)
    df['title_text']    = df.apply(build_title_text, axis=1)
    df['language_text'] = df.apply(build_language_text, axis=1)

print('Text features built for both 8K and 200K')

Text features built for both 8K and 200K


## 9. TF-IDF + SVD - fit on the UNION of 8K + 200K



In [9]:
text_channels = {
    'genre':    ('genre_text',    TfidfVectorizer(max_features=3000)),
    'people':   ('people_text',   TfidfVectorizer(max_features=4000)),
    'overview': ('overview_text', TfidfVectorizer(max_features=6000)),
    'title':    ('title_text',    TfidfVectorizer(max_features=2000, analyzer='char_wb', ngram_range=(3, 5))),
    'language': ('language_text', TfidfVectorizer(max_features=200)),
}

svd_components = {'genre': 50, 'people': 80, 'overview': 120, 'title': 40, 'language': 2}

tfidf_models = {}
svd_models = {}
emb_8k = {}
emb_200k = {}

n8k = len(movies_8k)

for name, (col, vectorizer) in text_channels.items():
    combined_text = pd.concat([movies_8k[col], movies_200k[col]], ignore_index=True)
    combined_vec = vectorizer.fit_transform(combined_text)          # fit on UNION

    svd = TruncatedSVD(n_components=svd_components[name], random_state=42)
    combined_emb = svd.fit_transform(combined_vec)                  # fit on UNION

    emb_8k[name]   = combined_emb[:n8k]
    emb_200k[name] = combined_emb[n8k:]

    tfidf_models[name] = vectorizer
    svd_models[name] = svd

    print(f'  {name:10s}: tfidf={combined_vec.shape}  svd={combined_emb.shape}')

print('TF-IDF + SVD fit on the union of both catalogs')

  genre     : tfidf=(243963, 3000)  svd=(243963, 50)
  people    : tfidf=(243963, 4000)  svd=(243963, 80)
  overview  : tfidf=(243963, 6000)  svd=(243963, 120)
  title     : tfidf=(243963, 2000)  svd=(243963, 40)
  language  : tfidf=(243963, 3)  svd=(243963, 2)
TF-IDF + SVD fit on the union of both catalogs


## 10. Numeric Features - content-only


In [10]:
numeric_cols = [
    'vote_average', 'vote_count', 'weighted_rating',   # IMDB WR - works for all 282K
    'revenue', 'runtime', 'popularity',
    'release_year', 'decade',
]

# Log-transform heavy-tailed columns BEFORE scaling, so outlier blockbusters don't
# get an enormous z-score that dominates the numeric channel.
skewed_cols = ['vote_count', 'revenue', 'popularity']
for df in (movies_8k, movies_200k):
    for col in skewed_cols:
        df[col] = np.log1p(df[col].clip(lower=0))

movies_8k[numeric_cols]   = movies_8k[numeric_cols].fillna(0)
movies_200k[numeric_cols] = movies_200k[numeric_cols].fillna(0)

combined_numeric = pd.concat([movies_8k[numeric_cols], movies_200k[numeric_cols]], ignore_index=True)
scaler = StandardScaler()
combined_numeric_scaled = scaler.fit_transform(combined_numeric)   # fit on UNION

numeric_8k   = combined_numeric_scaled[:n8k]
numeric_200k = combined_numeric_scaled[n8k:]

print(f'Numeric matrix (content-only, {len(numeric_cols)} features, log-transformed) fit on union')

Numeric matrix (content-only, 8 features, log-transformed) fit on union


## 11. Assemble CONTENT Embeddings

In [11]:
WEIGHT_GENRE    = 1.2
WEIGHT_PEOPLE   = 1.0
WEIGHT_OVERVIEW = 1.5
WEIGHT_TITLE    = 0.6
WEIGHT_LANGUAGE = 1.4
WEIGHT_NUMERIC  = 1.8

def assemble(emb_dict, numeric_scaled):
    return np.hstack([
        normalize(emb_dict['genre'])    * WEIGHT_GENRE,
        normalize(emb_dict['people'])   * WEIGHT_PEOPLE,
        normalize(emb_dict['overview']) * WEIGHT_OVERVIEW,
        normalize(emb_dict['title'])    * WEIGHT_TITLE,
        normalize(emb_dict['language']) * WEIGHT_LANGUAGE,
        normalize(numeric_scaled)       * WEIGHT_NUMERIC,
    ]).astype(np.float32)

content_emb_8k   = normalize(assemble(emb_8k, numeric_8k))
content_emb_200k = normalize(assemble(emb_200k, numeric_200k))

print(f'Content embeddings - 8K: {content_emb_8k.shape}, 200K: {content_emb_200k.shape}')

Content embeddings - 8K: (8248, 300), 200K: (235715, 300)


## 12. Deduplicate and Merge into a single catalog


In [12]:
movies_8k['has_real_ratings']   = True
movies_200k['has_real_ratings'] = False

all_movies_raw = pd.concat([movies_8k, movies_200k], ignore_index=True)
all_content_emb_raw = np.vstack([content_emb_8k, content_emb_200k])

all_movies_raw['_sort'] = all_movies_raw['has_real_ratings'].astype(int)
all_movies_raw = all_movies_raw.sort_values(['id', '_sort', 'vote_count'], ascending=[True, False, False])
all_movies_raw['_emb_idx'] = all_movies_raw.index

dedup_mask   = ~all_movies_raw.duplicated(subset=['id'], keep='first')
keep_emb_idx = all_movies_raw[dedup_mask]['_emb_idx'].tolist()

all_movies       = all_movies_raw[dedup_mask].drop(columns=['_sort', '_emb_idx']).reset_index(drop=True)
all_content_emb  = all_content_emb_raw[keep_emb_idx]

print(f'After dedup: {len(all_movies):,} rows, content embeddings: {all_content_emb.shape}')
print(f'  8K (with ratings)  : {all_movies["has_real_ratings"].sum():,}')
print(f'  200K (content only): {(~all_movies["has_real_ratings"]).sum():,}')

After dedup: 235,715 rows, content embeddings: (235715, 300)
  8K (with ratings)  : 8,248
  200K (content only): 227,467


## 13. Build the CONTENT FAISS Index (candidate generation for all 282K movies)

In [13]:
dim = all_content_emb.shape[1]
content_index = faiss.IndexFlatIP(dim)
content_index.add(all_content_emb)
print(f'Content FAISS index ready - {content_index.ntotal:,} vectors, dim={dim}')

def collapse_check(n_queries=300, k=10):
    counter = Counter()
    sample_idx = random.sample(range(len(all_movies)), n_queries)
    for qi in sample_idx:
        q_vec = all_content_emb[[qi]].astype(np.float32)
        _, neighbor_idx = content_index.search(q_vec, k + 1)
        counter.update(int(i) for i in neighbor_idx[0][:k] if i != qi)
    top_offenders = counter.most_common(10)
    distinct_hit_rate = len(counter) / (n_queries * k) * 100
    print(f'Distinct movies appearing in {n_queries} random top-{k} lists: {len(counter):,} '
          f'({distinct_hit_rate:.1f}% of the {n_queries * k} total slots)')
    print('Most frequently retrieved movies (title, times appeared, source):')
    for idx, count in top_offenders:
        row = all_movies.iloc[idx]
        print(f'  {count:4d}x  {row["title"]}  ({"8k" if row["has_real_ratings"] else "200k"})')
    return counter

_ = collapse_check()

Content FAISS index ready - 235,715 vectors, dim=300
Distinct movies appearing in 300 random top-10 lists: 2,674 (89.1% of the 3000 total slots)
Most frequently retrieved movies (title, times appeared, source):
     2x  Hitler's Madman  (200k)
     2x  Raise a Glass to Love  (200k)
     2x  Zolotoye Dno  (200k)
     2x  A Nag in the Bag  (200k)
     2x  The Hand of Fate; or The Mysterious Blonde  (200k)
     2x  The Wake  (200k)
     2x  Do You Remember Love  (200k)
     2x  Poets Are the Destroyers  (200k)
     2x  Happygram  (200k)
     2x  Tenis  (200k)


## 14. LightFM - bridging 8K collaborative signal to the full 282K catalog



In [ ]:
LIKE_THRESHOLD = 4.0  # single source of truth: matches build_query_indices and the eval "hit" definition

if USE_LIGHTFM:
    from lightfm.data import Dataset as LightFMDataset

    # Sparsify the (already dense, normalized) content embedding for LightFM's expected format.
    item_features_all = csr_matrix(all_content_emb)

    lightfm_dataset = LightFMDataset()
    lightfm_dataset.fit(
        users=ratings['userId'].unique(),
        items=all_movies['movieId'].dropna().unique(),
    )

    # Positive (liked) ratings only - WARP treats every row here as an implicit positive,
    # so mixing 1-star and 5-star ratings would train "any engagement", not "liked".
    ratings_positive = ratings[ratings['rating'] >= LIKE_THRESHOLD]
    ratings_valid = ratings_positive[ratings_positive['movieId'].isin(all_movies['movieId'].dropna())]

    n_all_valid = ratings[ratings['movieId'].isin(all_movies['movieId'].dropna())].shape[0]
    print(f'Positive-only interactions: {len(ratings_valid):,} (all valid ratings before filter: {n_all_valid:,})')

    (interactions, weights) = lightfm_dataset.build_interactions(
        [(row.userId, row.movieId, row.rating) for row in ratings_valid.itertuples()]
    )

    # Build an item_features matrix aligned to the SAME item ordering LightFM used internally.
    lightfm_item_id_map = lightfm_dataset.mapping()[2]     # {movieId: internal_item_idx}
    n_lightfm_items = len(lightfm_item_id_map)

    movieid_to_all_idx = {
        mid: idx for idx, mid in all_movies['movieId'].dropna().astype(int).items()
    }
    lightfm_features_rows = np.zeros((n_lightfm_items, all_content_emb.shape[1]), dtype=np.float32)
    for movie_id, lf_idx in lightfm_item_id_map.items():
        row_idx = movieid_to_all_idx.get(int(movie_id))
        if row_idx is not None:
            lightfm_features_rows[lf_idx] = all_content_emb[row_idx]
    lightfm_item_features = csr_matrix(lightfm_features_rows)

    lightfm_model = LightFM(loss='warp', no_components=64, random_state=42)
    lightfm_model.fit(interactions, item_features=lightfm_item_features, sample_weight=weights, epochs=30, num_threads=2)

    print(f'LightFM trained on {interactions.shape[0]:,} users x {interactions.shape[1]:,} items '
          f'(features shared across all {len(all_movies):,} catalog movies)')
else:
    lightfm_model = None
    lightfm_item_features = None
    lightfm_item_id_map = {}
    print('Skipping LightFM (not installed) - reranking will use content-similarity only.')

Positive-only interactions: 45,138 (all valid ratings before filter: 95,363)


In [ ]:
if USE_LIGHTFM:

    # ============================================================
    # PRODUCTION LIGHTFM
    # ============================================================
    # Trained on ALL available positive interactions. Keep this model for the
    # actual production recommender. Evaluation must NOT use these embeddings
    # because they contain the held-out interactions (see leakage-free block below).
    #
    # Row order note: `full_catalog_features` is passed in `all_movies` row order, not
    # LightFM's internal item order (`lightfm_item_id_map`). This is safe ONLY because
    # LightFM's item representation here is purely feature-space: with item_features
    # supplied, get_item_representations() returns (item_features @ feature_embeddings)
    # + feature_biases, computed independently per row with no dependency on the
    # internal item index or identity embeddings. Any query row of the feature matrix
    # gets the representation implied by its own features, regardless of position. If
    # this notebook is ever changed to use LightFM without item_features (pure identity
    # embeddings), this row-order assumption would break and must be revisited.

    full_catalog_features = csr_matrix(all_content_emb)

    lightfm_item_biases, lightfm_item_embeddings = (
        lightfm_model.get_item_representations(features=full_catalog_features)
    )

    print(f'Production LightFM collaborative-aware embeddings computed for all '
          f'{lightfm_item_embeddings.shape[0]:,} movies (dim={lightfm_item_embeddings.shape[1]})')

else:
    lightfm_item_embeddings = None
    lightfm_item_biases = None

movieid_to_idx = {
    int(row['movieId']): i for i, row in all_movies[all_movies['has_real_ratings']].iterrows()
    if pd.notna(row['movieId'])
}
user_histories = defaultdict(list)
for row in ratings.itertuples():
    if row.movieId in movieid_to_idx:
        user_histories[row.userId].append((row.movieId, row.rating, movieid_to_idx[row.movieId]))

eligible_users = {uid: h for uid, h in user_histories.items() if len(h) >= 10}

def ndcg_at_k(recommended, relevant_set, k):
    dcg = sum((1 / np.log2(i + 2)) for i, item in enumerate(recommended[:k]) if item in relevant_set)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

def build_query_indices(train_hist, like_threshold=LIKE_THRESHOLD):
    '''Only use movies the user actually liked to build their taste profile -
    averaging in disliked movies dilutes the signal.'''
    liked = [h for h in train_hist if h[1] >= like_threshold]
    if not liked:
        liked = [max(train_hist, key=lambda h: h[1])]   # fall back to their single favorite
    return [h[2] for h in liked]

def pick_held_out(hist, uid, seed_base=42):
    '''Deterministic held-out selection with reproducible tie-breaking.

    hist is sorted or unsorted list of (movieId, rating, idx). Ties on the max rating are
    broken by a per-user seeded RNG (seed depends on uid, not on call order or dict
    iteration), so every call site that needs "this user's held-out item" - the LightFM
    leakage exclusion set, evaluate_full_catalog, evaluate_sampled_negatives - agrees on
    the same item. This replaces the old `max(hist, key=lambda h: h[1])`, which broke
    ties by first-occurrence order and gave no guarantee that independent call sites
    picked the same item.
    '''
    best_rating = max(h[1] for h in hist)
    tied = [h for h in hist if h[1] == best_rating]
    if len(tied) == 1:
        return tied[0]
    rng = random.Random(hash((seed_base, uid)) & 0xFFFFFFFF)
    return rng.choice(tied)

def minmax_norm(x):
    '''Per-query min-max normalization - the SAME normalization production uses in
    recommend_from_indices. Evaluation functions use this too (fix for problem #1:
    prod and eval used to be scored on different scales).'''
    x = np.asarray(x, dtype=np.float64)
    if x.max() - x.min() < 1e-9:
        return np.zeros_like(x)
    return (x - x.min()) / (x.max() - x.min())

print('Helper functions ready (pick_held_out, minmax_norm shared by prod + eval)')

# NOTE on scope: movieid_to_idx / eligible_users are built only from has_real_ratings==True
# movies. Every user-level accuracy metric computed later in this notebook (HR@k, NDCG,
# the alpha sweep) is therefore a WARM-catalog metric only. It cannot and does not validate
# ranking quality on the 200K cold-start catalog - that requires the separate
# cold-start alpha-sensitivity check in section 20C.

### Leakage-free LightFM model for evaluation

Same positive-only fix applied here as in the production block above, so the evaluation
model is trained on the same objective ("liked", not "engaged with") that production uses.
Held-out pairs now use `pick_held_out()` so this exclusion set is guaranteed to match the
held-out items used later in `evaluate_full_catalog` and `evaluate_sampled_negatives`
(checked with an explicit assertion after this cell).

In [ ]:
if USE_LIGHTFM:

    EVAL_SEED = 42

    # --------------------------------------------------------
    # Identify the exact user/movie pairs held out by our
    # leave-one-out evaluation. Uses the shared pick_held_out()
    # helper so this exactly matches what evaluate_full_catalog
    # and evaluate_sampled_negatives will later hold out.
    # --------------------------------------------------------
    held_out_pairs = set()
    for uid, hist in eligible_users.items():
        top = pick_held_out(hist, uid, seed_base=EVAL_SEED)
        held_out_pairs.add((uid, top[0]))

    print(f'Excluding {len(held_out_pairs):,} held-out (user, movie) pairs from LightFM evaluation training')

    # --------------------------------------------------------
    # Positive-only ratings, then restrict to valid movies, then drop held-out pairs.
    # --------------------------------------------------------
    ratings_valid_eval = ratings[
        (ratings['rating'] >= LIKE_THRESHOLD) &
        (ratings['movieId'].isin(all_movies['movieId'].dropna()))
    ].copy()

    ratings_valid_eval['pair'] = list(zip(ratings_valid_eval['userId'], ratings_valid_eval['movieId']))
    ratings_valid_eval = (
        ratings_valid_eval[~ratings_valid_eval['pair'].isin(held_out_pairs)]
        .drop(columns=['pair'])
    )

    print(f'LightFM evaluation interactions (positive-only): {len(ratings_valid_eval):,} '
          f'(production positive-only interactions: {len(ratings_valid):,})')

    # --------------------------------------------------------
    # Build evaluation LightFM dataset
    # --------------------------------------------------------
    lightfm_dataset_eval = LightFMDataset()
    lightfm_dataset_eval.fit(
        users=ratings['userId'].unique(),
        items=all_movies['movieId'].dropna().unique(),
    )

    (interactions_eval, weights_eval) = lightfm_dataset_eval.build_interactions(
        [(row.userId, row.movieId, row.rating) for row in ratings_valid_eval.itertuples()]
    )

    # --------------------------------------------------------
    # Map all catalog movies into LightFM's item space
    # --------------------------------------------------------
    lightfm_item_id_map_eval = lightfm_dataset_eval.mapping()[2]
    n_lightfm_items_eval = len(lightfm_item_id_map_eval)

    lightfm_features_rows_eval = np.zeros((n_lightfm_items_eval, all_content_emb.shape[1]), dtype=np.float32)
    for movie_id, lf_idx in lightfm_item_id_map_eval.items():
        row_idx = movieid_to_all_idx.get(int(movie_id))
        if row_idx is not None:
            lightfm_features_rows_eval[lf_idx] = all_content_emb[row_idx]
    lightfm_item_features_eval = csr_matrix(lightfm_features_rows_eval)

    # --------------------------------------------------------
    # Train leakage-free evaluation model
    # --------------------------------------------------------
    lightfm_model_eval = LightFM(loss='warp', no_components=64, random_state=EVAL_SEED)
    lightfm_model_eval.fit(
        interactions_eval,
        item_features=lightfm_item_features_eval,
        sample_weight=weights_eval,
        epochs=30,
        num_threads=2
    )

    # --------------------------------------------------------
    # Generate representations for the FULL catalog
    # --------------------------------------------------------
    full_catalog_features = csr_matrix(all_content_emb)
    (lightfm_item_biases_eval, lightfm_item_embeddings_eval) = (
        lightfm_model_eval.get_item_representations(features=full_catalog_features)
    )

    print(f'Leakage-free LightFM trained - embeddings ready for '
          f'{lightfm_item_embeddings_eval.shape[0]:,} movies (dim={lightfm_item_embeddings_eval.shape[1]})')

else:
    lightfm_model_eval = None
    lightfm_item_embeddings_eval = None
    lightfm_item_biases_eval = None
    held_out_pairs = set()

In [ ]:
_recomputed_held_out = {
    (uid, pick_held_out(hist, uid, seed_base=EVAL_SEED)[0])
    for uid, hist in eligible_users.items()
}
assert _recomputed_held_out == held_out_pairs, (
    "Held-out pair mismatch between LightFM leakage exclusion and pick_held_out() - "
    "evaluation held-out items may not match what was excluded from training."
)
print(f'Held-out pair consistency check passed ({len(held_out_pairs):,} pairs agree).')

## 15. Smart Movie Lookup

In [ ]:
def find_movie(query_title, n=10, cutoff=0.4, verbose=True):
    all_titles = all_movies['title'].dropna().tolist()
    close = get_close_matches(query_title, all_titles, n=n, cutoff=cutoff)
    if not close:
        if verbose:
            print(f'No match for "{query_title}".')
        return pd.DataFrame()
    matched = all_movies[all_movies['title'].isin(close)].copy()
    return matched.sort_values('vote_count', ascending=False)

def load_movie(query_title, year=None, cutoff=0.4, verbose=True):
    candidates = find_movie(query_title, cutoff=cutoff, verbose=verbose)
    if candidates.empty:
        return None
    if year is not None:
        yr_filt = candidates[candidates['release_year'] == float(year)]
        if not yr_filt.empty:
            candidates = yr_filt
        elif verbose:
            print(f'Year {year} not found - using best vote_count match instead')
    best = candidates.iloc[0]
    if verbose:
        yr  = int(best['release_year']) if not pd.isna(best['release_year']) else '?'
        src = '8K+ratings' if best['has_real_ratings'] else '200K'
        print(f'  "{best["title"]}" ({yr}) | lang={best["original_language"]} '
              f'| votes={int(best["vote_count"])} | WR={best["weighted_rating"]:.2f} | source={src}')
    return best.name

def load_movies(title_list, year=None, verbose=True):
    return [i for t in title_list for i in [load_movie(t, year=year, verbose=verbose)] if i is not None]

print('Lookup functions defined')

## 16. Hybrid Recommender - Retrieve (content FAISS) then Rerank (LightFM)

1. Retrieve: top `k x N` candidates by content-only cosine similarity (works for all 282K movies).
2. Rerank: blend content similarity with the LightFM collaborative-aware score.
3. Quality filter: IMDB `weighted_rating`, uniform across both catalogs.

A single global `ALPHA_CONTENT` cannot serve both catalogs well. The alpha that maximizes
warm-item HR@10 (low alpha, more LightFM weight) lets LightFM's un-validated embedding
dominate cold-item ranking almost completely - at alpha=0.1 only 8% of a cold item's
top-10 neighbors survive from content-only ranking, vs. 86% at alpha=0.9 (see section
20C). `ALPHA_CONTENT` is replaced with a per-candidate alpha, chosen by whether the
candidate has real rating interactions (`has_real_ratings`). Warm items lean on LightFM
(informed by the bootstrap-confirmed HR@10 sweep in section 20B); cold items lean on
content, since their LightFM embedding is pure feature-extrapolation that has never been
checked against a real interaction (see section 14).

Blending uses `minmax_norm()` - the same per-query min-max normalization used everywhere
in evaluation (section 20) as of this notebook, so `alpha` means the same thing in
production and in every evaluation harness below.

In [ ]:
ALPHA_WARM = 0.1   # movies with real ratings.csv interactions - trust LightFM more (see section 20B)
ALPHA_COLD = 0.8   # 200K-only movies - LightFM embedding is pure extrapolation, lean on content

def segment_alpha(candidate_idx, alpha_warm=ALPHA_WARM, alpha_cold=ALPHA_COLD):
    '''Per-candidate alpha: has_real_ratings drives how much to trust LightFM vs content.'''
    has_ratings = all_movies.iloc[candidate_idx]['has_real_ratings'].to_numpy()
    return np.where(has_ratings, alpha_warm, alpha_cold)

def recommend_from_indices(liked_indices, k=10, min_weighted_rating=5.5,
                            same_language=False, same_era=False, era_window=15,
                            candidate_multiplier=25):
    if not liked_indices:
        print('No valid indices')
        return pd.DataFrame()

    input_rows  = all_movies.iloc[liked_indices]
    input_langs = set(input_rows['original_language'].dropna().tolist())
    input_years = input_rows['release_year'].dropna().tolist()
    year_mean   = float(np.mean(input_years)) if input_years else None

    # --- Stage 1: per-item retrieval + max-fusion union ---
    n_candidates = k * candidate_multiplier
    per_item_k = max(n_candidates // max(len(liked_indices), 1), 50)
    candidate_pool = {}
    for qi in liked_indices:
        qv = all_content_emb[[qi]].astype(np.float32).copy()
        faiss.normalize_L2(qv)
        scores, idxs = content_index.search(qv, per_item_k)
        for s, i in zip(scores[0], idxs[0]):
            if i not in candidate_pool or s > candidate_pool[i]:
                candidate_pool[i] = s
    candidate_idx = np.array(list(candidate_pool.keys()))
    content_scores = np.array([candidate_pool[i] for i in candidate_idx])

    # --- Stage 2: LightFM collaborative rerank score ---
    if USE_LIGHTFM and lightfm_item_embeddings is not None:
        lightfm_query = lightfm_item_embeddings[liked_indices].mean(axis=0)
        lightfm_raw = (lightfm_item_biases[candidate_idx] +
                       lightfm_item_embeddings[candidate_idx] @ lightfm_query)
        lightfm_norm = minmax_norm(lightfm_raw)
    else:
        lightfm_norm = np.zeros(len(candidate_idx))

    content_norm = minmax_norm(content_scores)
    alpha_vec = segment_alpha(candidate_idx)          # per-candidate, not global
    blended = alpha_vec * content_norm + (1 - alpha_vec) * lightfm_norm

    results = []
    order = np.argsort(-blended)
    for rank_pos in order:
        i = candidate_idx[rank_pos]
        if i in liked_indices:
            continue
        row = all_movies.iloc[i]

        if row['weighted_rating'] < min_weighted_rating:
            continue
        if same_language and row['original_language'] not in input_langs:
            continue
        if same_era and year_mean is not None:
            yr = row['release_year']
            if pd.isna(yr) or abs(yr - year_mean) > era_window:
                continue

        results.append({
            'title'           : row['title'],
            'year'            : int(row['release_year']) if not pd.isna(row['release_year']) else '?',
            'language'        : row['original_language'],
            'weighted_rating' : round(float(row['weighted_rating']), 2),
            'votes'           : int(row['vote_count']),
            'content_score'   : round(float(content_norm[rank_pos]), 4),
            'lightfm_score'   : round(float(lightfm_norm[rank_pos]), 4),
            'alpha_used'      : round(float(alpha_vec[rank_pos]), 2),
            'blended_score'   : round(float(blended[rank_pos]), 4),
            'source'          : '8k' if row['has_real_ratings'] else '200k',
        })
        if len(results) == k:
            break

    df = pd.DataFrame(results)
    if df.empty:
        print('No results - try relaxing min_weighted_rating or filter options.')
    return df

def recommend(title_list, k=10, min_weighted_rating=5.5, same_language=False,
              same_era=False, era_window=15, year=None):
    print('Resolving input movies...')
    indices = load_movies(title_list, year=year)
    if not indices:
        print('Could not resolve any titles.')
        return pd.DataFrame()
    print(f'Generating top {k} hybrid recommendations...')
    return recommend_from_indices(indices, k=k, min_weighted_rating=min_weighted_rating,
                                   same_language=same_language, same_era=same_era, era_window=era_window)

print('Hybrid recommender functions defined (segmented alpha, shared minmax_norm)')

## 17. Discovery & Browsing

In [ ]:
def show_high_rated_random(n=20, min_weighted_rating=6.5, language=None, year_from=None, year_to=None, seed=None):
    df = all_movies.copy()
    df = df[df['weighted_rating'] >= min_weighted_rating]
    if language:
        df = df[df['original_language'] == language]
    if year_from:
        df = df[df['release_year'] >= year_from]
    if year_to:
        df = df[df['release_year'] <= year_to]
    if seed is not None:
        random.seed(seed)
    sample = df.sample(n=min(n, len(df)))

    header = f'High-Rated Movies (weighted_rating >= {min_weighted_rating}'
    if language: header += f', lang={language}'
    if year_from or year_to: header += f', {year_from or "?"}-{year_to or "?"}'
    print(header + ')\n')

    for i, (_, row) in enumerate(sample.iterrows(), 1):
        yr = int(row['release_year']) if not pd.isna(row['release_year']) else '?'
        src = '8k' if row['has_real_ratings'] else '200k'
        print(f"{i:02d}. [{row['original_language']:2s}] {row['title']} ({yr})"
              f"  WR={row['weighted_rating']:.2f}  ({src})")
    return sample

print('Discovery functions defined')

## 18. Usage Examples

In [ ]:
print('=' * 65); print('TEST 1: Cheaper by the Dozen (2003) - family comedy'); print('=' * 65)
recommend(['Cheaper by the Dozen'], year=2003)

print('=' * 65); print('TEST 2: Parasite - Korean language filter'); print('=' * 65)
recommend(['Parasite'], same_language=True, k=10)

print('=' * 65); print('TEST 3: Titanic - era filter +/- 15 years'); print('=' * 65)
recommend(['Titanic'], same_era=True, era_window=15, k=10)

print('=' * 65); print('TEST 4: Browse - Hindi films 2000-2023'); print('=' * 65)
show_high_rated_random(n=10, min_weighted_rating=6.0, language='hi', year_from=2000, year_to=2023)

## 19. Evaluation - Part A: Intrinsic content-embedding quality

In [ ]:
def extract_primary_genre(genres_str):
    if pd.isna(genres_str): return None
    s = str(genres_str)
    if s.startswith('['):
        try:
            parsed = ast.literal_eval(s)
            if parsed:
                item = parsed[0]
                return item.get('name', str(item)) if isinstance(item, dict) else str(item)
        except Exception:
            pass
    parts = [g.strip() for g in s.replace('|', ',').split(',') if g.strip()]
    return parts[0] if parts else None

all_movies['primary_genre'] = all_movies['genres_y'].apply(extract_primary_genre)

def genre_cluster_purity(genre, n_sample=50, n_neighbors=10):
    genre_idx = all_movies[all_movies['primary_genre'] == genre].index.tolist()
    if len(genre_idx) < n_sample:
        return None, len(genre_idx)
    sampled = random.sample(genre_idx, n_sample)
    query_vecs = all_content_emb[sampled].astype(np.float32)
    _, neighbor_idx = content_index.search(query_vecs, n_neighbors + 1)
    purities = []
    for neighbors in neighbor_idx:
        neighbors = [n for n in neighbors if n < len(all_movies)]
        neighbor_genres = all_movies.iloc[neighbors[1:]]['primary_genre'].tolist()
        purities.append(sum(g == genre for g in neighbor_genres) / len(neighbor_genres))
    return np.mean(purities), len(genre_idx)

target_genres = ['Drama', 'Comedy', 'Action', 'Thriller', 'Horror', 'Romance', 'Science Fiction', 'Animation', 'Documentary']
purity_results = [dict(genre=g, purity=p, n_movies=n)
                   for g in target_genres
                   for p, n in [genre_cluster_purity(g)] if p is not None]
purity_df = pd.DataFrame(purity_results).sort_values('purity', ascending=False)
print(f'Mean genre cluster purity: {purity_df["purity"].mean():.3f}')
purity_df

## 20. Evaluation - Part B: Warm-catalog accuracy (leave-one-out)



In [ ]:
EVAL_SEED = 42
_REF_SAMPLE_SIZE = 5000
_POOL_SIZE = 1000

_eval_rng = random.Random(EVAL_SEED)
FIXED_EVAL_USERS = _eval_rng.sample(list(eligible_users.keys()), min(300, len(eligible_users)))
print(f'Fixed evaluation users: {len(FIXED_EVAL_USERS):,}')

def evaluate_full_catalog(use_lightfm_rerank, eval_users, alpha=0.6, use_segment_alpha=False,
                           pool_size=1000, k_values=(5, 10, 20), like_threshold=LIKE_THRESHOLD,
                           item_biases=None, item_embeddings=None, min_weighted_rating=None):
    '''
    Full-catalog evaluation.

    alpha: scalar blend weight, used unless use_segment_alpha=True.
    use_segment_alpha: if True, ignore `alpha` and use segment_alpha() (has_real_ratings-based)
        instead - lets this same function validate the segmented-alpha production path.
    min_weighted_rating: if set, candidates below this weighted_rating are dropped before
        ranking - reproduces production's quality filter inside evaluation (fix #5). None
        (default) reproduces the old, unfiltered behavior for backward comparison.

    Scoring uses minmax_norm() throughout - identical to production's
    recommend_from_indices (fix #1), so `alpha` means the same thing here as it does in
    the actual serving path.

    IMPORTANT: when LightFM is enabled, leakage-free evaluation embeddings MUST be supplied.
    '''
    out = {k: {'hit': [], 'ndcg': []} for k in k_values}
    recall_at_pool = []

    if use_lightfm_rerank and USE_LIGHTFM:
        if item_biases is None or item_embeddings is None:
            raise ValueError(
                "Leakage-free LightFM embeddings are required for evaluation. Pass "
                "item_biases=lightfm_item_biases_eval and item_embeddings=lightfm_item_embeddings_eval."
            )

    for uid in eval_users:
        hist = list(eligible_users[uid])
        held_out = pick_held_out(hist, uid, seed_base=EVAL_SEED)
        train_hist = [h for h in hist if h != held_out]

        if held_out[1] < 4.0 or not train_hist:
            continue

        query_idx = build_query_indices(train_hist, like_threshold)
        if len(query_idx) == 0:
            continue

        per_item_k = max(pool_size // len(query_idx), 50)
        query_vectors = all_content_emb[query_idx].astype(np.float32).copy()
        faiss.normalize_L2(query_vectors)
        scores, idxs = content_index.search(query_vectors, per_item_k)

        candidate_pool = {}
        for row_scores, row_idxs in zip(scores, idxs):
            for score, movie_idx in zip(row_scores, row_idxs):
                if movie_idx not in candidate_pool or score > candidate_pool[movie_idx]:
                    candidate_pool[movie_idx] = score

        if not candidate_pool:
            continue

        candidate_idx = np.fromiter(candidate_pool.keys(), dtype=np.int64)
        content_scores = np.fromiter(candidate_pool.values(), dtype=np.float32)

        if min_weighted_rating is not None:
            wr = all_movies.iloc[candidate_idx]['weighted_rating'].to_numpy()
            keep = wr >= min_weighted_rating
            candidate_idx = candidate_idx[keep]
            content_scores = content_scores[keep]
            if len(candidate_idx) == 0:
                continue

        recall_at_pool.append(int(held_out[2] in set(candidate_idx.tolist())))

        if use_lightfm_rerank and USE_LIGHTFM:
            lightfm_query = item_embeddings[query_idx].mean(axis=0)
            lf_raw = item_biases[candidate_idx] + item_embeddings[candidate_idx] @ lightfm_query

            content_norm = minmax_norm(content_scores)
            lightfm_norm = minmax_norm(lf_raw)

            if use_segment_alpha:
                alpha_vec = segment_alpha(candidate_idx)
            else:
                alpha_vec = alpha

            blended = alpha_vec * content_norm + (1.0 - alpha_vec) * lightfm_norm
        else:
            blended = minmax_norm(content_scores)

        seen = {h[2] for h in train_hist}
        order = np.argsort(-blended)
        recs = [candidate_idx[i] for i in order if candidate_idx[i] not in seen]
        relevant = {held_out[2]}

        for k in k_values:
            hit = int(bool(set(recs[:k]) & relevant))
            out[k]['hit'].append(hit)
            out[k]['ndcg'].append(ndcg_at_k(recs, relevant, k))

    out['recall_at_pool'] = np.mean(recall_at_pool) if recall_at_pool else 0.0
    out['n_users_scored'] = len(out[k_values[0]]['hit'])
    return out

print('Evaluation infra ready (shared pick_held_out + minmax_norm, optional quality filter)')

results_content_only = evaluate_full_catalog(
    use_lightfm_rerank=False, eval_users=FIXED_EVAL_USERS, pool_size=_POOL_SIZE
)

results_hybrid = evaluate_full_catalog(
    use_lightfm_rerank=True, eval_users=FIXED_EVAL_USERS, pool_size=_POOL_SIZE,
    item_biases=lightfm_item_biases_eval, item_embeddings=lightfm_item_embeddings_eval
)

print()
print(f"Recall@{_POOL_SIZE} (ceiling on the comparison below - see fix #7 note above):")
print(f"  content-only={results_content_only['recall_at_pool']:.4f}  "
      f"hybrid={results_hybrid['recall_at_pool']:.4f}")
print()

for k in (5, 10, 20):
    content_hr = np.mean(results_content_only[k]['hit'])
    content_ndcg = np.mean(results_content_only[k]['ndcg'])
    hybrid_hr = np.mean(results_hybrid[k]['hit'])
    hybrid_ndcg = np.mean(results_hybrid[k]['ndcg'])
    print(f"k={k:2d} | content-only  HR@k={content_hr:.4f}  NDCG@k={content_ndcg:.4f}"
          f"   |   hybrid  HR@k={hybrid_hr:.4f}  NDCG@k={hybrid_ndcg:.4f}")

### 20A. Production-filtered comparison 

In [ ]:
PRODUCTION_MIN_WR = 5.5

results_content_only_filtered = evaluate_full_catalog(
    use_lightfm_rerank=False, eval_users=FIXED_EVAL_USERS, pool_size=_POOL_SIZE,
    min_weighted_rating=PRODUCTION_MIN_WR
)
results_hybrid_filtered = evaluate_full_catalog(
    use_lightfm_rerank=True, eval_users=FIXED_EVAL_USERS, pool_size=_POOL_SIZE,
    item_biases=lightfm_item_biases_eval, item_embeddings=lightfm_item_embeddings_eval,
    min_weighted_rating=PRODUCTION_MIN_WR
)

print(f'At production filter (weighted_rating >= {PRODUCTION_MIN_WR}):')
print(f"  users scored: content-only={results_content_only_filtered['n_users_scored']}, "
      f"hybrid={results_hybrid_filtered['n_users_scored']} (out of {len(FIXED_EVAL_USERS)} eval users - "
      f"a user drops out if every retrieved candidate falls below the quality bar)")
for k in (5, 10, 20):
    c = np.mean(results_content_only_filtered[k]['hit'])
    h = np.mean(results_hybrid_filtered[k]['hit'])
    print(f"  k={k:2d} | content-only HR@k={c:.4f}  |  hybrid HR@k={h:.4f}")

### 20B. Alpha sweep with bootstrap confidence intervals



In [ ]:
def bootstrap_hr_ci(hits, n_boot=500, seed=42):
    hits = np.array(hits)
    rng = np.random.RandomState(seed)
    boots = [hits[rng.randint(0, len(hits), len(hits))].mean() for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return hits.mean(), lo, hi

print('Sweeping alpha (warm-catalog eval, leakage-free LightFM, positive-only interactions, minmax_norm)...\n')

alpha_results_clean = {}
for alpha in (0.1, 0.3, 0.5, 0.6, 0.7, 0.9):
    res = evaluate_full_catalog(
        use_lightfm_rerank=True, eval_users=FIXED_EVAL_USERS, alpha=alpha, pool_size=1000,
        item_biases=lightfm_item_biases_eval, item_embeddings=lightfm_item_embeddings_eval
    )
    alpha_results_clean[alpha] = res
    mean, lo, hi = bootstrap_hr_ci(res[10]['hit'])
    print(f'alpha={alpha:.1f} | HR@5={np.mean(res[5]["hit"]):.4f}  '
          f'HR@10={mean:.4f} (95% CI [{lo:.4f}, {hi:.4f}])  '
          f'HR@20={np.mean(res[20]["hit"]):.4f}')

### 20B (cont). Leave-One-Out with sampled negatives - popularity vs quality vs content vs hybrid baselines


In [ ]:
all_8k_idx = list(movieid_to_idx.values())

def evaluate_sampled_negatives(scoring_fn, k_values=(5, 10, 20), n_eval=300, n_negatives=99, seed=42):
    '''scoring_fn(uid, hist, train_hist, candidates) -> np.array of scores, one per candidate'''
    rng = random.Random(seed)
    eval_users = rng.sample(list(eligible_users.keys()), min(n_eval, len(eligible_users)))
    out = {k: {'hit': [], 'ndcg': []} for k in k_values}
    for uid in eval_users:
        hist = list(eligible_users[uid])
        held_out = pick_held_out(hist, uid, seed_base=EVAL_SEED)
        train_hist = [h for h in hist if h != held_out]
        if held_out[1] < 4.0 or not train_hist:
            continue
        seen_idx = {h[2] for h in hist}
        pool = [i for i in all_8k_idx if i not in seen_idx]
        negatives = rng.sample(pool, min(n_negatives, len(pool)))
        candidates = negatives + [held_out[2]]
        rng.shuffle(candidates)

        scores = scoring_fn(uid, hist, train_hist, candidates)
        order = np.argsort(-scores)
        ranked = [candidates[i] for i in order]
        rank_of_positive = ranked.index(held_out[2])

        for k in k_values:
            hit = int(rank_of_positive < k)
            out[k]['hit'].append(hit)
            out[k]['ndcg'].append(1 / np.log2(rank_of_positive + 2) if hit else 0.0)
    return out

def content_scoring(uid, hist, train_hist, candidates, use_lightfm=False):
    query_idx = build_query_indices(train_hist)
    query_vec = all_content_emb[query_idx].mean(axis=0)
    scores = all_content_emb[candidates] @ query_vec

    if use_lightfm and USE_LIGHTFM:
        candidates_arr = np.array(candidates)
        # Use the leakage-free evaluation LightFM model - never the production embeddings here.
        lf_query = lightfm_item_embeddings_eval[query_idx].mean(axis=0)
        lf_scores = (lightfm_item_biases_eval[candidates_arr]
                     + lightfm_item_embeddings_eval[candidates_arr] @ lf_query)

        # Same blend production uses: segment_alpha() + minmax_norm() (fix #6).
        alpha_vec = segment_alpha(candidates_arr)
        scores = alpha_vec * minmax_norm(scores) + (1 - alpha_vec) * minmax_norm(lf_scores)

    return scores

def quality_baseline_scoring(uid, hist, train_hist, candidates):
    '''Renamed from popularity_scoring - this is a QUALITY baseline (IMDB weighted
    rating), not a popularity baseline. See popularity_baseline_scoring below for the
    latter.'''
    return all_movies.iloc[candidates]['weighted_rating'].values

def popularity_baseline_scoring(uid, hist, train_hist, candidates):
    '''A genuine popularity baseline: number of ratings received (interaction volume),
    not rating quality. Falls back to log(vote_count) for any candidate without a
    rating_count (not expected in this 8K-only evaluation loop, but kept for safety).'''
    rc = all_movies.iloc[candidates]['rating_count'].values
    vc = all_movies.iloc[candidates]['vote_count'].values  # already log1p-transformed in section 10
    return np.where(rc > 0, rc, vc)

res_pop     = evaluate_sampled_negatives(popularity_baseline_scoring)
res_quality = evaluate_sampled_negatives(quality_baseline_scoring)
res_content = evaluate_sampled_negatives(lambda *a: content_scoring(*a, use_lightfm=False))
res_hybrid  = evaluate_sampled_negatives(lambda *a: content_scoring(*a, use_lightfm=True))

print(f'{"k":>4} | {"popularity HR/NDCG":>20} | {"quality HR/NDCG":>18} | {"content HR/NDCG":>18} | {"hybrid HR/NDCG":>16}')
for k in (5, 10, 20):
    print(f'{k:4d} | {np.mean(res_pop[k]["hit"]):>9.4f} / {np.mean(res_pop[k]["ndcg"]):<7.4f}'
          f' | {np.mean(res_quality[k]["hit"]):>8.4f} / {np.mean(res_quality[k]["ndcg"]):<7.4f}'
          f' | {np.mean(res_content[k]["hit"]):>8.4f} / {np.mean(res_content[k]["ndcg"]):<7.4f}'
          f' | {np.mean(res_hybrid[k]["hit"]):>6.4f} / {np.mean(res_hybrid[k]["ndcg"]):<7.4f}')

## 20C. Evaluation - Part C: Cold-start alpha sensitivity


In [ ]:
def coldstart_alpha_sensitivity(n_sample=30, k=10, alphas=(0.1, 0.3, 0.6, 0.9), pool=200, seed=42):
    rng = random.Random(seed)
    cold_idx = all_movies.index[~all_movies['has_real_ratings']].tolist()
    sample = rng.sample(cold_idx, n_sample)
    for alpha in alphas:
        overlaps = []
        for qi in sample:
            qv = all_content_emb[[qi]].astype(np.float32).copy()
            faiss.normalize_L2(qv)
            # Search pool+1 and drop the query item itself (self-match at similarity 1.0),
            # matching collapse_check()'s convention - fix #2.
            _, idxs = content_index.search(qv, pool + 1)
            idxs = np.array([i for i in idxs[0] if i != qi])[:pool]

            content_scores = all_content_emb[idxs] @ all_content_emb[qi]
            lf_q = lightfm_item_embeddings[qi]
            lf_raw = lightfm_item_biases[idxs] + lightfm_item_embeddings[idxs] @ lf_q

            content_norm = minmax_norm(content_scores)
            lightfm_norm = minmax_norm(lf_raw)
            blended = alpha * content_norm + (1 - alpha) * lightfm_norm

            top_blend   = set(idxs[np.argsort(-blended)[:k]])
            top_content = set(idxs[np.argsort(-content_scores)[:k]])
            overlaps.append(len(top_blend & top_content) / k)
        print(f'alpha={alpha:.1f} | avg overlap with content-only top-{k} on cold items = {np.mean(overlaps):.2f}')

print('Cold-start sensitivity across candidate alphas (self-match excluded):')
coldstart_alpha_sensitivity()

print('\nCold-start sensitivity check at the production ALPHA_COLD value:')
coldstart_alpha_sensitivity(alphas=(ALPHA_COLD,))
# With the self-match bug fixed, re-derive what "healthy" overlap looks like at ALPHA_COLD
# empirically from the sweep above rather than assuming the old 0.7-0.8 bound still holds -
# that bound was measured under the buggy version and is not guaranteed to transfer.

### Validating the segmented-alpha production path end to end, with a paired-bootstrap CI (fix #9)


In [ ]:
results_segmented = evaluate_full_catalog(
    use_lightfm_rerank=True, eval_users=FIXED_EVAL_USERS, use_segment_alpha=True, pool_size=_POOL_SIZE,
    item_biases=lightfm_item_biases_eval, item_embeddings=lightfm_item_embeddings_eval
)

mean, lo, hi = bootstrap_hr_ci(results_segmented[10]['hit'])
print(f'Segmented alpha (production path) | HR@10={mean:.4f} (95% CI [{lo:.4f}, {hi:.4f}])  '
      f'HR@5={np.mean(results_segmented[5]["hit"]):.4f}  HR@20={np.mean(results_segmented[20]["hit"]):.4f}')
print(f'For comparison, fixed alpha=ALPHA_WARM={ALPHA_WARM}: '
      f'HR@10={np.mean(alpha_results_clean[ALPHA_WARM][10]["hit"]):.4f}')

def paired_bootstrap_diff_ci(hits_a, hits_b, n_boot=2000, seed=42):
    '''Paired bootstrap CI on mean(hits_a) - mean(hits_b). Valid only when hits_a[i] and
    hits_b[i] come from the SAME user i in the SAME order - true here because both runs
    iterate FIXED_EVAL_USERS identically and skip users based only on user data, not alpha.'''
    a = np.array(hits_a)
    b = np.array(hits_b)
    assert len(a) == len(b), "Hit arrays must be paired (same users, same order) for this CI to be valid."
    rng = np.random.RandomState(seed)
    diffs = []
    n = len(a)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        diffs.append(a[idx].mean() - b[idx].mean())
    diffs = np.array(diffs)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return diffs.mean(), lo, hi

point_diff, diff_lo, diff_hi = paired_bootstrap_diff_ci(
    results_segmented[10]['hit'], alpha_results_clean[ALPHA_WARM][10]['hit']
)
verdict = (
    "segmented alpha is not distinguishable from fixed ALPHA_WARM at HR@10 (CI includes 0)"
    if diff_lo <= 0 <= diff_hi else
    "segmented alpha significantly differs from fixed ALPHA_WARM at HR@10"
)
print(f'\nHR@10 difference (segmented - fixed ALPHA_WARM) = {point_diff:+.4f}  '
      f'95% CI [{diff_lo:+.4f}, {diff_hi:+.4f}]')
print(f'-> {verdict}.')
print('Segmenting is still justified independently by the cold-start check in section 20C: '
      'even where it does not move warm HR@10, it materially changes cold-item ranking '
      'quality, which this warm-only metric cannot see.')

## 20D. Evaluation - Part D: Beyond-accuracy, catalog coverage

In [ ]:
def compute_coverage(n_queries=300, k=10):
    recommended_set = set()
    sample_idx = random.sample(range(len(all_movies)), n_queries)
    for qi in sample_idx:
        q_vec = all_content_emb[[qi]].astype(np.float32)
        _, neighbor_idx = content_index.search(q_vec, k + 1)
        recommended_set.update(neighbor_idx[0][:k])
    return (len(recommended_set) / len(all_movies)) * 100

print(f'Catalog Coverage: {compute_coverage():.2f}%')

## 21. Save Artifacts


In [ ]:
import pickle, os, json
os.makedirs('artifacts', exist_ok=True)

pickle.dump(scaler, open('artifacts/scaler.pkl', 'wb'))

for name, model in tfidf_models.items():
    pickle.dump(model, open(f'artifacts/tfidf_{name}.pkl', 'wb'))
for name, model in svd_models.items():
    pickle.dump(model, open(f'artifacts/svd_{name}.pkl', 'wb'))

if USE_LIGHTFM:
    pickle.dump(lightfm_model, open('artifacts/lightfm_model.pkl', 'wb'))
    np.save('artifacts/lightfm_item_embeddings.npy', lightfm_item_embeddings)
    np.save('artifacts/lightfm_item_biases.npy', lightfm_item_biases)

with open('artifacts/blend_config.json', 'w') as f:
    json.dump({
        'ALPHA_WARM': ALPHA_WARM,
        'ALPHA_COLD': ALPHA_COLD,
        'LIKE_THRESHOLD': LIKE_THRESHOLD,
        'PRODUCTION_MIN_WR': PRODUCTION_MIN_WR,
    }, f, indent=2)

faiss.write_index(content_index, 'artifacts/content_index.faiss')
all_movies.to_parquet('artifacts/all_movies.parquet', index=False)
np.save('artifacts/all_content_embeddings.npy', all_content_emb)

print('Artifacts saved to /artifacts/')

In [ ]:
# import shutil
# from google.colab import files

# # Zip the artifacts directory
# shutil.make_archive('artifacts', 'zip', 'artifacts')

# # Automatically download to your device
# files.download('artifacts.zip')